# LoRA

这个实操的目标是改编 [Karpathy](https://karpathy.ai/) 的 [minGPT](https://github.com/karpathy/minGPT/) 代码，把低秩适配（Low Rank Adaptation，LoRA）加进去做微调。

![](https://miro.medium.com/v2/resize:fit:720/format:webp/1*D_i25E9dTd_5HMa45zITSg.png)

[Rajan Ghimire](https://r4j4n.github.io/blogs/about/) 的这篇[博客](https://r4j4n.github.io/blogs/posts/lora/) 是一篇不错的 LoRA 入门介绍。


In [ ]:
import math
from dataclasses import dataclass

import torch
import torch.nn as nn
from torch.nn import functional as F

## 构建自定义 Linear 模块

方法
- [`forward`](https://pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.forward)
- [`train`](https://pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.train)
- [`eval`](https://pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.eval)
- [`reset_parameters`](https://github.com/pytorch/pytorch/blob/v2.6.0/torch/nn/modules/linear.py#L50)


In [ ]:
class LoRALinear(nn.Linear):

    def __init__(self,
                 # nn.Linear 参数
                 in_features: int,
                 out_features: int,
                 bias: bool = True,
                 device=None,
                 dtype=None,
                 # LoRA 参数
                 lora_rank: int = 0,
                 lora_alpha: float = 0.0,
                ) -> None:
        nn.Linear.__init__(
            self,
            in_features=in_features,
            out_features=out_features,
            bias=bias,
            device=device,
            dtype=dtype
        )

        # LoRA 相关
        self.has_weights_merged = False
        if lora_rank > 0:
            self.lora_scaling = lora_alpha / lora_rank
            self.lora_A = nn.Parameter(torch.empty((lora_rank, self.in_features), device=device, dtype=dtype))
            self.lora_B = nn.Parameter(torch.empty((self.out_features, lora_rank), device=device, dtype=dtype))

            self.lora_A.requires_grad = False
            self.lora_B.requires_grad = False

            self.reset_parameters_lora()


    def reset_parameters_lora(self) -> None:
        #####
        # 你的代码
        #####

    def forward(self, input: torch.Tensor) -> torch.Tensor:
        x = nn.Linear.forward(self, input)
        #####
        # 你的代码
        #####
        return x

    def train(self, mode: bool = True) -> "LoRALinear":
        nn.Linear.train(self, mode)
        #####
        # 你的代码
        #####
        return self

    def eval(self) -> "LoRALinear":
        nn.Linear.eval(self)
        #####
        # 你的代码
        #####
        return self

In [ ]:
ln = LoRALinear(in_features=3,out_features=4, lora_rank = 8, lora_alpha = 32)

In [ ]:
ln.weight

In [ ]:
ln.bias

In [ ]:
for p in ln.parameters():
    print(p)

In [ ]:
bs = 5
x = torch.randn((bs, 3))
y = ln(x)

In [ ]:
y2 = x@ln.weight.T + ln.bias

In [ ]:
torch.isclose(y,y2)

In [ ]:
ln.train()

In [ ]:
y3 = ln(x)
torch.isclose(y3,y2)

In [ ]:
ln.eval()
y3 = ln(x)
torch.isclose(y3,y2)

In [ ]:
def get_lora_model(model: nn.Module) -> nn.Module:
    for name, param in model.named_parameters():
        if "lora" in name:
            param.requires_grad = True
        else:
            param.requires_grad = False
    return model

In [ ]:
ln_lora = get_lora_model(ln)

In [ ]:
for p in ln_lora.parameters():
    print(p)

## 在 minGPT 的构建块里使用 LoRA 层


In [ ]:
from mingpt.model import CausalSelfAttention

class CausalSelfAttention_LoRA(CausalSelfAttention):
    def __init__(self, config):
        super().__init__(config)
        # 小修改
        self.c_attn = LoRALinear(
            in_features=config.n_embd,
            out_features=3 * config.n_embd,
            lora_rank=config.lora_rank,
            lora_alpha=config.lora_alpha,
        )
        # 输出投影
        self.c_proj = LoRALinear(
            in_features=config.n_embd,
            out_features=config.n_embd,
            lora_rank=config.lora_rank,
            lora_alpha=config.lora_alpha,
        )

In [ ]:
from mingpt.model import Block, NewGELU

class Block_LoRA(Block):
    """ an unassuming Transformer block """

    def __init__(self, config):
        super().__init__(config)
        # 小修改
        self.attn = CausalSelfAttention_LoRA(config)

GPT 模块也一样，而且你可以简化 LoRA 模块的优化器配置


In [ ]:
from mingpt.model import GPT

class GPT_LoRA(GPT):
    def __init__(self, config):
        super().__init__(config)
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            drop = nn.Dropout(config.embd_pdrop),
            h = nn.ModuleList([Block_LoRA(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        self.config = config
        # 初始化所有权重，并按 GPT-2 论文对残差投影做特殊的缩放初始化
        self.apply(self._init_weights)
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02/math.sqrt(2 * config.n_layer))

    def configure_optimizers(self, train_config):
        """
        This long function is unfortunately doing something very simple and is being very defensive:
        We are separating out all parameters of the model into two buckets: those that will experience
        weight decay for regularization and those that won't (biases, and layernorm/embedding weights).
        We are then returning the PyTorch optimizer object.
        """

        #####
        # 你的代码
        #####
        
        # 把参数分成需要权重衰减和不需要权重衰减的两组
        decay = set()
        no_decay = set()
        whitelist_weight_modules = (torch.nn.Linear, )
        blacklist_weight_modules = (torch.nn.LayerNorm, torch.nn.Embedding)
        for mn, m in self.named_modules():
            for pn, p in m.named_parameters():
                fpn = '%s.%s' % (mn, pn) if mn else pn  # 完整参数名
                # 随机备注：由于 named_modules 和 named_parameters 是递归的
                # 同一个张量 p 我们会看到很多很多次。但这样做
                # 让我们知道任何张量 p 属于哪个父模块……
                if pn.endswith('bias'):
                    # 所有偏置都不做衰减
                    no_decay.add(fpn)
                elif pn.endswith('weight') and isinstance(m, whitelist_weight_modules):
                    # 白名单模块的权重会做权重衰减
                    decay.add(fpn)
                elif pn.endswith('weight') and isinstance(m, blacklist_weight_modules):
                    # 黑名单模块的权重不会做权重衰减
                    no_decay.add(fpn)

        # 验证我们考虑到了每个参数
        param_dict = {pn: p for pn, p in self.named_parameters()}
        inter_params = decay & no_decay
        union_params = decay | no_decay
        assert len(inter_params) == 0, "parameters %s made it into both decay/no_decay sets!" % (str(inter_params), )
        assert len(param_dict.keys() - union_params) == 0, "parameters %s were not separated into either decay/no_decay set!" \
                                                    % (str(param_dict.keys() - union_params), )

        # 创建 pytorch 优化器对象
        optim_groups = [
            {"params": [param_dict[pn] for pn in sorted(list(decay))], "weight_decay": train_config.weight_decay},
            {"params": [param_dict[pn] for pn in sorted(list(no_decay))], "weight_decay": 0.0},
        ]
        optimizer = torch.optim.AdamW(optim_groups, lr=train_config.learning_rate, betas=train_config.betas)
        return optimizer

## 学习排序

我们用[这个 demo](https://github.com/karpathy/minGPT/blob/master/demo.ipynb) 来确认我们的代码运行正常！


In [ ]:
@dataclass
class Config:
    n_head = 3
    n_embd = 15
    block_size = 11
    # dropout 超参数
    embd_pdrop = 0.1
    resid_pdrop = 0.1
    attn_pdrop = 0.1
    # LoRA
    lora_rank = 8
    lora_alpha = 32

# 创建一个 GPT 实例
model_config = GPT.get_default_config()
model_config.model_type = 'gpt-nano'
model_config.vocab_size = 3
model_config.block_size = 100
model_config.lora_rank = 8
model_config.lora_alpha = 32

model = GPT_LoRA(model_config)

In [ ]:
from torch.utils.data import Dataset
from torch.utils.data.dataloader import DataLoader
from mingpt.utils import set_seed
set_seed(3407)
import pickle

class SortDataset(Dataset):
    """ 
    Dataset for the Sort problem. E.g. for problem length 6:
    Input: 0 0 2 1 0 1 -> Output: 0 0 0 1 1 2
    Which will feed into the transformer concatenated as:
    input:  0 0 2 1 0 1 0 0 0 1 1
    output: I I I I I 0 0 0 1 1 2
    where I is "ignore", as the transformer is reading the input sequence
    """

    def __init__(self, split, length=6, num_digits=3):
        assert split in {'train', 'test'}
        self.split = split
        self.length = length
        self.num_digits = num_digits
    
    def __len__(self):
        return 10000  # ...
    
    def get_vocab_size(self):
        return self.num_digits
    
    def get_block_size(self):
        # 喂给 transformer 的序列长度，
        # 包含拼接后的输入和输出，但减 1，因为
        # transformer 从最后一个输入元素开始做预测
        return self.length * 2 - 1

    def __getitem__(self, idx):
        
        # 用拒绝采样从想要的划分生成一个输入样本
        while True:
            # 生成一些随机整数
            inp = torch.randint(self.num_digits, size=(self.length,), dtype=torch.long)
            # 一半时间尝试增加重复数多的样本数量，
            # 因为模型在训练后期似乎难以处理这些，
            # 而且它们比较稀有
            if torch.rand(1).item() < 0.5:
                if inp.unique().nelement() > self.length // 2:
                    # 唯一数字太多，重新采样
                    continue
            # 根据哈希判断这个生成的样本属于训练集还是测试集
            h = hash(pickle.dumps(inp.tolist()))
            inp_split = 'test' if h % 4 == 0 else 'train' # designate 25% of examples as test
            if inp_split == self.split:
                break  # 好
        
        # 解决任务：也就是排序
        sol = torch.sort(inp)[0]

        # 拼接问题描述和解答
        cat = torch.cat((inp, sol), dim=0)

        # 喂给 transformer 的输入是偏移后的序列
        x = cat[:-1].clone()
        y = cat[1:].clone()
        # 我们只想在输出位置做预测，把输入位置的损失掩码掉
        y[:self.length-1] = -1
        return x, y

In [ ]:
# 打印数据集的一个示例实例
train_dataset = SortDataset('train')
test_dataset = SortDataset('test')

In [ ]:
model_config = GPT.get_default_config()
model_config.model_type = 'gpt-nano'
model_config.vocab_size = train_dataset.get_vocab_size()
model_config.block_size = 24  # train_dataset.get_block_size()
model_config.lora_rank = 8
model_config.lora_alpha = 32
model_config.lora_dropout = 0
model = GPT_LoRA(model_config)

In [ ]:
# 创建一个 Trainer 对象
from mingpt.trainer import Trainer

train_config = Trainer.get_default_config()
train_config.learning_rate = 5e-4  # 我们用的模型非常小，可以稍微快一点
train_config.max_iters = 1000  # 2000
train_config.num_workers = 0
trainer = Trainer(train_config, model, train_dataset)

In [ ]:
def batch_end_callback(trainer):
    if trainer.iter_num % 100 == 0:
        print(f"iter_dt {trainer.iter_dt * 1000:.2f}ms; iter {trainer.iter_num}: train loss {trainer.loss.item():.5f}")
trainer.set_callback('on_batch_end', batch_end_callback)

trainer.run()

In [ ]:
# 现在做一些评估
model.eval();
dataset = {'train':train_dataset, 'test':test_dataset}
def eval_split(trainer, split, max_batches, dataset=dataset):
    dataset = dataset[split]
    n = dataset.length  # 直接访问，有点调皮，耸肩
    results = []
    mistakes_printed_already = 0
    loader = DataLoader(dataset, batch_size=100, num_workers=0, drop_last=False)
    for b, (x, y) in enumerate(loader):
        x = x.to(trainer.device)
        y = y.to(trainer.device)
        # 单独取出输入模式
        inp = x[:, :n]
        sol = y[:, -n:]
        # 让模型采样序列的其余部分
        cat = model.generate(inp, n, do_sample=False)  # 用贪心 argmax，不是采样
        sol_candidate = cat[:, -n:]  # 取出填充后的序列
        # 把预测序列和真实序列比较
        correct = (sol == sol_candidate).all(1).cpu()  # Software 1.0 与 Software 2.0 就在这一行开战哈哈
        for i in range(x.size(0)):
            results.append(int(correct[i]))
            if not correct[i] and mistakes_printed_already < 3:  # 只打印最多 5 个错误，了解一下情况
                mistakes_printed_already += 1
                print("GPT claims that %s sorted is %s but gt is %s" % (inp[i].tolist(), sol_candidate[i].tolist(), sol[i].tolist()))
        if max_batches is not None and b+1 >= max_batches:
            break
    rt = torch.tensor(results, dtype=torch.float)
    print("%s final score: %d/%d = %.2f%% correct" % (split, rt.sum(), len(results), 100*rt.mean()))
    return rt.sum()

# 让大量训练集和测试集样本通过模型，验证输出正确性
with torch.no_grad():
    train_score = eval_split(trainer, 'train', max_batches=50)
    test_score  = eval_split(trainer, 'test',  max_batches=50)

现在我们稍微修改数据集的分布，用 LoRA 来做微调。


In [ ]:
train_dataset2 = SortDataset('train',length=10)
test_dataset2 = SortDataset('test',length=10)

In [ ]:
dataset2 = {'train':train_dataset2, 'test':test_dataset2}
with torch.no_grad():
    train_score = eval_split(trainer, 'train', max_batches=50, dataset=dataset2)
    test_score  = eval_split(trainer, 'test',  max_batches=50, dataset=dataset2)

In [ ]:
# 在这里写用 LoRA 训练的代码

In [ ]:
model.eval();
with torch.no_grad():
    train_score = eval_split(trainer, 'train', max_batches=50, dataset=dataset2)
    test_score  = eval_split(trainer, 'test',  max_batches=50, dataset=dataset2)
